# GMAI-Pulse — GWAM Canada Retirement · Interactive Charts

A **modern, dark-theme, interactive** companion to the profiling notebook
(`gwam_canada_retirement_eda.py`). Where the EDA notebook emits machine-readable JSON,
this notebook renders the same meaningful observations as **Plotly** charts you can
hover, zoom, and filter — built for a **global team**.

### What makes it global-team friendly
- **Time-zone selector** — every time-of-day chart is recomputed in the reviewer's own
  timezone (hits are converted from `hit_time_gmt` epoch-GMT), so a viewer in Toronto,
  London, Hong Kong, or Manila sees activity aligned to *their* clock.
- **Geography selector** — filter to a country / region, or view the world choropleth and
  province bars, to see where CA-Retirement traffic originates (CAN ~85 %, USA ~11 %).
- **Date-range & granularity** — daily or weekly rollups over any sub-window.

### Privacy (ADR-0007)
Every panel is an **aggregate** (counts / rates by time or by an allow-listed dimension).
No visitor IDs, IPs, cookies, user-agents, or fine geo are ever read or displayed.

### How to run
Attach to a cluster (DBR 13+; Plotly ships with DBR-ML). Run the **Config** cell, adjust the
widgets that appear at the top, then Run All. Each chart cell is independent — re-run one
after changing a widget. Colour palette is CVD-validated (dataviz skill, dark surface).

## Config, widgets, theme & helpers

In [0]:
from pyspark.sql import functions as F
import json, math, hashlib

# Plotly ships with Databricks Runtime for ML. If you are on a plain runtime, uncomment:
# %pip install -q plotly
import plotly.graph_objects as go
import plotly.io as pio

# ---------------------------------------------------------------- widgets ----
dbutils.widgets.text("table_fqn", "gwam_prod_catalog.inv_typed_common.adobe_hit_data", "1. Table (catalog.schema.table)")
dbutils.widgets.text("rsid_filter", "manulifeglobalprod", "2. rsid filter (empty = off)")
dbutils.widgets.text("url_filter", "manulife.com/ca/en/personal/group-plans/group-retirement", "3. URL contains filter (empty = off)")
dbutils.widgets.text("start_date", "2026-02-01", "4. Start date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2026-07-07", "5. End date (YYYY-MM-DD)")
dbutils.widgets.dropdown("geo_country", "ALL",
                         ["ALL", "can", "usa", "hkg", "phl", "ind", "gbr"], "6. Country (geo_country)")
dbutils.widgets.text("geo_region", "", "7. Region(s), comma-sep e.g. on,qc (empty = all)")
dbutils.widgets.dropdown("timezone", "America/Toronto",
                         ["America/Toronto", "America/Vancouver", "America/New_York",
                          "America/Sao_Paulo", "Europe/London", "Europe/Paris",
                          "Asia/Hong_Kong", "Asia/Manila", "Asia/Kolkata",
                          "Asia/Tokyo", "Australia/Sydney"], "8. Reviewer timezone")
dbutils.widgets.dropdown("granularity", "daily", ["daily", "weekly"], "9. Time granularity")
dbutils.widgets.text("top_n", "12", "10. Top-N for dimension bars")

TABLE_FQN   = dbutils.widgets.get("table_fqn").strip()
RSID_FILTER = dbutils.widgets.get("rsid_filter").strip().lower()
URL_FILTER  = dbutils.widgets.get("url_filter").strip().lower()
START_DATE  = dbutils.widgets.get("start_date").strip()
END_DATE    = dbutils.widgets.get("end_date").strip()
GEO_COUNTRY = dbutils.widgets.get("geo_country").strip().lower()
GEO_REGIONS = [r.strip().lower() for r in dbutils.widgets.get("geo_region").split(",") if r.strip()]
TIMEZONE    = dbutils.widgets.get("timezone").strip()
GRANULARITY = dbutils.widgets.get("granularity").strip().lower()
TOP_N       = int(dbutils.widgets.get("top_n"))

# ---------------------------------------------------- dark theme (dataviz) ----
# Categorical hues: the dataviz reference dark palette (validated CVD, surface #1a1a19).
# Fixed order — assigned by entity, never cycled.
CATEGORICAL = ["#3987e5", "#199e70", "#c98500", "#008300",
               "#9085e9", "#e66767", "#d55181", "#d95926"]
# Sequential blue for magnitude (heatmap / choropleth), stepped for the DARK surface:
# near-zero recedes toward the surface, high values brighten.
SEQ_BLUE = [[0.0, "#10233d"], [0.25, "#184f95"], [0.5, "#256abf"],
            [0.75, "#3987e5"], [1.0, "#86b6ef"]]
INK, INK2, SURFACE, PLANE, GRID = "#ffffff", "#c3c2b7", "#1a1a19", "#0d0d0d", "#2c2c2a"

pio.templates["gmai_dark"] = go.layout.Template(layout=dict(
    paper_bgcolor=PLANE, plot_bgcolor=SURFACE,
    font=dict(color=INK2, family='"Segoe UI", system-ui, sans-serif', size=13),
    colorway=CATEGORICAL,
    title=dict(font=dict(color=INK, size=18), x=0.01, xanchor="left"),
    xaxis=dict(gridcolor=GRID, zerolinecolor="#383835", linecolor="#383835", ticks="outside"),
    yaxis=dict(gridcolor=GRID, zerolinecolor="#383835", linecolor="#383835", rangemode="tozero"),
    legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", yanchor="bottom", y=1.02, x=0),
    margin=dict(l=60, r=30, t=70, b=50), hovermode="x unified",
))
pio.templates.default = "plotly_dark+gmai_dark"

def render(fig, height=460):
    """Render interactively in Databricks (displayHTML) or Jupyter (fig.show())."""
    fig.update_layout(height=height)
    try:
        displayHTML(fig.to_html(include_plotlyjs="cdn", full_html=False))  # noqa: F821
    except NameError:
        fig.show()

# ------------------------------------------------------------ data helpers ----
def nonblank(c):
    col = F.col(c)
    return col.isNotNull() & (F.trim(col.cast("string")) != "")

def local_ts():
    """hit_time_gmt is epoch-seconds GMT -> convert to the reviewer's TIMEZONE.
    Falls back to date_time (already Eastern local) if hit_time_gmt is absent."""
    cols = set(base_df.columns)
    if "hit_time_gmt" in cols:
        return F.from_utc_timestamp(F.timestamp_seconds(F.col("hit_time_gmt").cast("long")), TIMEZONE)
    return F.to_timestamp(F.col("date_time"))

def trunc_period(ts_col):
    return F.date_trunc("week", ts_col) if GRANULARITY == "weekly" else F.to_date(ts_col)

# --------------------------------------------------- shareable data emit ----
# Mirrors the EDA notebook's emit() protocol: the exported .ipynb carries the
# aggregate data behind every chart as machine-readable JSON, so a run can be
# analysed offline without re-querying. Aggregate-only per ADR-0007.
CHART_DATA = {}

def _scrub(obj):
    """Round floats (4dp), truncate long strings — parity with the EDA scrubber."""
    if isinstance(obj, dict):
        return {k: _scrub(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_scrub(v) for v in obj]
    if isinstance(obj, str):
        return obj if len(obj) <= 160 else obj[:160] + "...<trunc>"
    if isinstance(obj, float):
        return round(obj, 4) if math.isfinite(obj) else None
    return obj

def _records(pdf, date_cols=()):
    """pandas frame -> list of plain-Python dicts (native int/float, ISO date strings).
    Round-tripping through to_json/loads drops numpy int64/float64 and Timestamps."""
    pdf = pdf.copy()
    for c in date_cols:
        if c in pdf.columns:
            pdf[c] = pdf[c].astype(str)
    return json.loads(pdf.to_json(orient="records"))

def share(chart_id, payload):
    """Print + register the aggregate behind a chart as a SHAREABLE JSON block."""
    payload = _scrub(payload)
    CHART_DATA[chart_id] = payload
    body = json.dumps(payload, separators=(",", ":"), default=str)
    sid = f"chart:{chart_id}"
    print(f"===== BEGIN SHAREABLE: {sid} =====")
    if len(body) <= 48000:
        print(body)
    else:
        n_parts = math.ceil(len(body) / 40000)
        for i in range(n_parts):
            print(f"----- part {i+1} of {n_parts} (concatenate parts to reassemble) -----")
            print(body[i * 40000:(i + 1) * 40000])
    print(f"===== END SHAREABLE: {sid} =====")

print(f"table={TABLE_FQN}  tz={TIMEZONE}  granularity={GRANULARITY}")
print(f"scope: rsid={RSID_FILTER or '(off)'}  url~{URL_FILTER or '(off)'}  "
      f"country={GEO_COUNTRY}  regions={GEO_REGIONS or 'all'}  dates={START_DATE}..{END_DATE}")

table=gwam_prod_catalog.inv_typed_common.adobe_hit_data  tz=America/Toronto  granularity=daily
scope: rsid=manulifeglobalprod  url~manulife.com/ca/en/personal/group-plans/group-retirement  country=all  regions=all  dates=2026-02-01..2026-07-07


## Load & scope the CA-Retirement subset
Applies the report-suite + URL scope (same definition as the EDA notebook), then the
widget-driven country / region / date filters. `base_df` is reused by every chart.

In [0]:
_raw = spark.table(TABLE_FQN)
_cols = set(_raw.columns)

# --- scope: rsid == filter AND page URL contains filter (each optional) ---
_scope = None
if RSID_FILTER and "rsid" in _cols:
    _scope = (F.lower(F.trim(F.col("rsid").cast("string"))) == F.lit(RSID_FILTER))
_url_col = "post_page_url" if "post_page_url" in _cols else ("page_url" if "page_url" in _cols else None)
if URL_FILTER and _url_col:
    _u = F.lower(F.col(_url_col).cast("string")).contains(URL_FILTER)
    _scope = _u if _scope is None else (_scope & _u)
base_df = _raw.filter(_scope) if _scope is not None else _raw

# --- date range on the reviewer-local calendar date ---
base_df = base_df.filter((F.to_date(local_ts()) >= F.lit(START_DATE)) &
                         (F.to_date(local_ts()) <= F.lit(END_DATE)))

# --- geography filters ---
if GEO_COUNTRY != "all" and "geo_country" in _cols:
    base_df = base_df.filter(F.lower(F.col("geo_country")) == F.lit(GEO_COUNTRY))
if GEO_REGIONS and "geo_region" in _cols:
    base_df = base_df.filter(F.lower(F.col("geo_region")).isin(GEO_REGIONS))

base_df = base_df.cache()
_scoped_rows = base_df.count()
print(f"scoped rows in view: {_scoped_rows:,}")
if _scoped_rows == 0:
    print("!! 0 rows — widen the date range / clear the geo filters / check the scope widgets.")

share("scope", {"table": TABLE_FQN, "rsid_filter": RSID_FILTER or None,
                "url_filter": URL_FILTER or None, "start_date": START_DATE,
                "end_date": END_DATE, "geo_country": GEO_COUNTRY,
                "geo_regions": GEO_REGIONS or None, "timezone": TIMEZONE,
                "granularity": GRANULARITY, "top_n": TOP_N,
                "scoped_rows": _scoped_rows})

scoped rows in view: 1,151,474


## 1 · Traffic over time — hits, visits, visitors
**Form:** multi-series line (change over time). Weekly seasonality is the dominant feature
(weekends ≈ 40 % of weekdays); RRSP season (Feb–Mar) is the volume peak. Drag on the range
slider to zoom; toggle a series in the legend.

In [0]:
vis_hi = "post_visid_high" if "post_visid_high" in _cols else ("visid_high" if "visid_high" in _cols else None)
vis_lo = "post_visid_low" if "post_visid_low" in _cols else ("visid_low" if "visid_low" in _cols else None)

_aggs = [F.count("*").alias("hits")]
if vis_hi and vis_lo and "visit_num" in _cols:
    _aggs.append(F.approx_count_distinct(F.concat_ws(":", vis_hi, vis_lo, "visit_num")).alias("visits"))
    _aggs.append(F.approx_count_distinct(F.concat_ws(":", vis_hi, vis_lo)).alias("visitors"))

_ts = (base_df.groupBy(trunc_period(local_ts()).alias("period")).agg(*_aggs)
              .orderBy("period").toPandas())

share("traffic_ts", {"tz": TIMEZONE, "granularity": GRANULARITY,
                     "rows": _records(_ts, date_cols=["period"])})

fig = go.Figure()
_series = [("hits", CATEGORICAL[0]), ("visits", CATEGORICAL[1]), ("visitors", CATEGORICAL[3])]
for name, color in _series:
    if name in _ts.columns:
        fig.add_trace(go.Scatter(x=_ts["period"], y=_ts[name], name=name, mode="lines",
                                 line=dict(color=color, width=2),
                                 hovertemplate=f"%{{x|%a %Y-%m-%d}}<br>{name}: %{{y:,}}<extra></extra>"))
# direct end-labels (secondary encoding for the CVD floor band)
for name, color in _series:
    if name in _ts.columns and len(_ts):
        fig.add_annotation(x=_ts["period"].iloc[-1], y=_ts[name].iloc[-1], text=f" {name}",
                           showarrow=False, xanchor="left", font=dict(color=color, size=12))
fig.update_layout(title=f"CA-Retirement traffic ({GRANULARITY}) — {TIMEZONE}",
                  yaxis_title="count", xaxis_title=None)
fig.update_xaxes(rangeslider=dict(visible=True), rangeslider_thickness=0.06)
render(fig)

## 2 · When are users active? — day-of-week × hour heatmap
**Form:** sequential heatmap (magnitude). Computed in the **selected timezone**, so the
peak-hour band shifts as you change the widget. Under Eastern time the peak is ~10:00 on
weekdays; switch to `Asia/Hong_Kong` to see the same hits on that clock.

In [0]:
_hh = (base_df.select(F.dayofweek(local_ts()).alias("dow"), F.hour(local_ts()).alias("hr"))
              .groupBy("dow", "hr").count().toPandas())
# Spark dayofweek: 1=Sun..7=Sat -> reorder to Mon..Sun
_dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
_z = [[0] * 24 for _ in range(7)]
for _, r in _hh.iterrows():
    _z[(int(r["dow"]) + 5) % 7][int(r["hr"])] = int(r["count"])

share("dow_hour", {"tz": TIMEZONE, "dow_mon_to_sun": _dow_names,
                   "hours_0_23": [f"{h:02d}" for h in range(24)], "z": _z})

fig = go.Figure(go.Heatmap(
    z=_z, x=[f"{h:02d}" for h in range(24)], y=_dow_names, colorscale=SEQ_BLUE,
    colorbar=dict(title="hits", outlinewidth=0),
    hovertemplate="%{y} %{x}:00<br>hits: %{z:,}<extra></extra>"))
fig.update_layout(title=f"Activity by weekday × hour — {TIMEZONE}",
                  xaxis_title="hour of day", yaxis_title=None, hovermode="closest")
fig.update_yaxes(autorange="reversed")
render(fig, height=380)

## 3 · Where does traffic come from? — country choropleth
**Form:** sequential choropleth (magnitude by place). CA-Retirement is ~85 % Canada, ~11 %
USA, with a long tail (HKG / PHL / IND). Adobe stores ISO-3 lowercase codes; we upper-case
for the map. Hover a country for its hit count.

In [0]:
if "geo_country" in _cols:
    _gc = (base_df.filter(nonblank("geo_country"))
                  .groupBy(F.upper(F.col("geo_country")).alias("iso3")).count()
                  .orderBy(F.desc("count")).toPandas())
    share("geo_country", {"rows": _records(_gc)})
    fig = go.Figure(go.Choropleth(
        locations=_gc["iso3"], z=_gc["count"], locationmode="ISO-3",
        colorscale=SEQ_BLUE, colorbar=dict(title="hits", outlinewidth=0),
        marker_line_color="#2c2c2a",
        hovertemplate="%{location}<br>hits: %{z:,}<extra></extra>"))
    fig.update_layout(title="Traffic by country (geo_country)",
                      geo=dict(bgcolor=SURFACE, lakecolor=SURFACE,
                               showframe=False, showcoastlines=False,
                               projection_type="natural earth"))
    render(fig, height=460)
else:
    print("geo_country not present in this table build — skipping country map.")

## 4 · Top regions & pages
**Form:** horizontal magnitude bars (single hue — one measure). Canadian provinces
(`geo_region`: ON ~47 %, AB ~15 %, BC ~12 %, QC ~4 %) and the busiest pages
(`ca-ret:personal:overview` dominates).

In [0]:
def top_bar(col, title):
    if col not in _cols:
        print(f"{col} not present — skipping.")
        return
    pdf = (base_df.filter(nonblank(col)).groupBy(col).count()
                  .orderBy(F.desc("count")).limit(TOP_N).toPandas().iloc[::-1])
    share(f"top_{col}", {"top_n": TOP_N, "rows": _records(pdf)})
    fig = go.Figure(go.Bar(
        x=pdf["count"], y=pdf[col].astype(str), orientation="h",
        marker=dict(color="#3987e5"),
        text=pdf["count"].map(lambda v: f"{v:,}"), textposition="outside",
        hovertemplate="%{y}<br>hits: %{x:,}<extra></extra>"))
    fig.update_layout(title=title, xaxis_title="hits", yaxis_title=None, margin=dict(l=180))
    render(fig, height=max(320, 26 * len(pdf) + 120))

top_bar("geo_region", f"Top {TOP_N} regions (geo_region)")
top_bar("pagename", f"Top {TOP_N} pages (pagename)")

## 5 · Language mix over time
**Form:** stacked area (composition over time). Adobe numeric codes: **45 ≈ English**
(~63 %), **39 ≈ French** (~30 %); everything else grouped as *Other*. A stable mix is the
expected state — a sudden swing is worth investigating.

In [0]:
_LANG = {"45": "English (45)", "39": "French (39)"}
if "language" in _cols:
    lang_expr = F.col("language").cast("string")
    label = F.when(lang_expr == "45", F.lit(_LANG["45"])) \
             .when(lang_expr == "39", F.lit(_LANG["39"])).otherwise(F.lit("Other"))
    _lm = (base_df.filter(nonblank("language"))
                  .groupBy(trunc_period(local_ts()).alias("period"), label.alias("lang"))
                  .count().orderBy("period").toPandas())
    share("language_mix", {"granularity": GRANULARITY,
                           "rows": _records(_lm, date_cols=["period"])})
    _piv = _lm.pivot_table(index="period", columns="lang", values="count", fill_value=0)
    _order = [c for c in ["English (45)", "French (39)", "Other"] if c in _piv.columns]
    _colors = {"English (45)": CATEGORICAL[0], "French (39)": CATEGORICAL[1], "Other": "#898781"}
    fig = go.Figure()
    for name in _order:
        fig.add_trace(go.Scatter(x=_piv.index, y=_piv[name], name=name, mode="lines",
                                 stackgroup="one", line=dict(width=0.5, color=_colors[name]),
                                 fillcolor=_colors[name],
                                 hovertemplate=f"%{{x|%Y-%m-%d}}<br>{name}: %{{y:,}}<extra></extra>"))
    fig.update_layout(title=f"Language mix ({GRANULARITY})", yaxis_title="hits", xaxis_title=None)
    render(fig)
else:
    print("language not present — skipping language mix.")

## 6 · Event / KPI firing timeline
**Form:** multi-series line (change over time). Daily count of hits **carrying** each KPI
event. This panel makes the **instrumentation on/off shifts** visible: `ev501–504` start
**2026-02-24**; `ev500` fires **only 2026-04-02 → mid-May**. Those are known tagging
changes, *not* anomalies — the detector must treat them as change-points.

Event names are the Adobe platform defaults from `new_data/event.tsv` (20 = Campaign View;
500–504 = clickmap events). These IDs are fully resolved; only the eVar *content* meaning
(what each eVar captures) still needs the eVar dictionary.

In [0]:
# Names from new_data/event.tsv (standard Adobe event lookup).
_KPI_EVENTS = {"500": "ev500 clickmappage", "20": "ev20 campaign-view",
               "501": "ev501 clickmaplink", "502": "ev502 clickmapregion",
               "503": "ev503 clickmaplinkbyregion", "504": "ev504 targetsessionid"}
_ev_col = "post_event_list" if "post_event_list" in _cols else ("event_list" if "event_list" in _cols else None)
if _ev_col:
    ids = F.array_distinct(F.transform(
        F.filter(F.transform(F.split(F.col(_ev_col), ","), lambda x: F.trim(x)), lambda x: x != ""),
        lambda x: F.split(x, "=")[0]))
    _ev = (base_df.filter(nonblank(_ev_col))
                  .select(trunc_period(local_ts()).alias("period"), F.explode(ids).alias("eid"))
                  .filter(F.col("eid").isin(list(_KPI_EVENTS.keys())))
                  .groupBy("period", "eid").count().orderBy("period").toPandas())
    share("event_timeline", {"granularity": GRANULARITY, "event_names": _KPI_EVENTS,
                             "rows": _records(_ev, date_cols=["period"])})
    fig = go.Figure()
    for i, (eid, label) in enumerate(_KPI_EVENTS.items()):
        sub = _ev[_ev["eid"] == eid]
        if len(sub):
            fig.add_trace(go.Scatter(x=sub["period"], y=sub["count"], name=label, mode="lines",
                                     line=dict(color=CATEGORICAL[i % len(CATEGORICAL)], width=2),
                                     hovertemplate=f"%{{x|%Y-%m-%d}}<br>{label}: %{{y:,}} hits<extra></extra>"))
    fig.update_layout(title=f"KPI event firing ({GRANULARITY}) — hits carrying each event",
                      yaxis_title="hits with event", xaxis_title=None)
    render(fig)
else:
    print("post_event_list not present — skipping event timeline.")

## 7 · Monthly seasonality (RRSP season)
**Form:** magnitude bars by month. February–March (RRSP contribution season) is the peak;
volume tapers into summer. A month-over-month detector must expect this and not alarm on it.

In [0]:
_mo = (base_df.groupBy(F.date_format(local_ts(), "yyyy-MM").alias("month")).count()
              .orderBy("month").toPandas())
share("monthly_volume", {"rows": _records(_mo)})
fig = go.Figure(go.Bar(x=_mo["month"], y=_mo["count"], marker=dict(color="#3987e5"),
                       text=_mo["count"].map(lambda v: f"{v:,}"), textposition="outside",
                       hovertemplate="%{x}<br>hits: %{y:,}<extra></extra>"))
fig.update_layout(title="Monthly hit volume (RRSP seasonality)", yaxis_title="hits", xaxis_title=None)
render(fig, height=380)

## Data manifest — integrity check for the export
Byte length + sha1 of every `chart:<id>` block above, so a truncated `.ipynb`
export can be caught offline (re-hash a block, compare against this manifest).

In [ ]:
_manifest = {}
for _cid, _payload in CHART_DATA.items():
    _body = json.dumps(_payload, separators=(",", ":"), default=str)
    _manifest[_cid] = {"bytes": len(_body),
                       "sha1": hashlib.sha1(_body.encode("utf-8")).hexdigest()}
share("manifest", {"charts": _manifest, "n_charts": len(_manifest)})

---
**Notes.** Charts are aggregate-only (ADR-0007). Geography columns (`geo_country`,
`geo_region`) are populated in this source table but are **not yet in the production
pipeline bronze layer** — see `research/claude/12-eda-findings-analysis.md` §6 and
`databricks/conf/bronze_columns.py` if these need to reach the detector. Palette is the
CVD-validated dataviz dark set; multi-series panels carry a legend + hover + direct labels
so identity is never colour-alone.

In [0]:
base_df.unpersist()

DataFrame[accept_language: string, adclassificationcreative: string, adload: string, aemassetid: string, aemassetsource: string, aemclickedassetid: string, browser: string, browser_height: string, browser_width: string, c_color: string, campaign: string, carrier: string, ch_hdr: string, ch_js: string, channel: string, click_action: string, click_action_type: string, click_context: string, click_context_type: string, click_sourceid: string, click_tag: string, clickmaplink: string, clickmaplinkbyregion: string, clickmappage: string, clickmapregion: string, code_ver: string, color: string, connection_type: string, cookies: string, country: string, ct_connect_type: string, curr_factor: string, curr_rate: string, currency: string, cust_hit_time_gmt: string, cust_visid: string, daily_visitor: string, dataprivacyconsentoptin: string, dataprivacyconsentoptout: string, date_time: string, domain: string, duplicate_events: string, duplicate_purchase: string, duplicated_from: string, ef_id: string